# Arrhythmia (호환 약자) UMAP

이전 6-1 figure의 약자(`SA · ALS · APB · AF · SVT · TWC · ST`) 중
**ICD-10 매핑된 5개** (SA, APB, AF, SVT, ST) 를 cross-dataset 비교.

## 분석
1. 약자 → ICD 매핑 표 재출력
2. **PTB-XL + ZZU**에서 매핑된 5개 ICD 코드로 UMAP (combined / adult / pediatric)
3. **chapman** 소아 분포 확인 (성인 10,091 / 소아 141)
4. chapman 자체에서 7개 raw 약자 UMAP (chapman 단일)

In [ ]:
%load_ext autoreload
%autoreload 2
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from scripts.umap_view import (
    quick_dx, quick_dx_raw,
    ICD_LABEL_MAP, ICD_DISPLAY,
    list_models, get_ages,
)
import pandas as pd
import numpy as np
pd.set_option('display.width', 160)

## 1) 약자 → ICD 매핑 표

5/7 만 ICD_LABEL_MAP에 등록됨. ALS, TWC는 chapman 고유라 cross-dataset 매핑 없음.

In [ ]:
user_codes = [
    ('SA',  'Sinus arrhythmia',          None,   'SARRH', 'R00.8'),
    ('ALS', 'chapman 고유 (T-axis 변이)',  'ALS',  None,    None),
    ('APB', 'Atrial Premature Beat',     'APB',  'PAC',   'I49.1'),
    ('AF',  'Atrial Fibrillation',       'AF',   'AFIB',  'I48'),
    ('SVT', 'Supraventricular Tachy',    'SVT',  'PSVT',  'I47.1'),
    ('TWC', 'T Wave Change',             'TWC',  None,    None),
    ('ST',  'Sinus Tachycardia',         'ST',   'STACH', 'R00.0'),
]
rows = [{
    'short': s, 'meaning': m, 'chapman': c or '-', 'ptbxl': p or '-',
    'ICD': i or '-',
    'mapped': bool(p and ICD_LABEL_MAP['ptbxl'].get(p)),
    'display': ICD_DISPLAY.get(i, '-') if i else '-',
} for s, m, c, p, i in user_codes]
tbl = pd.DataFrame(rows)
print(tbl.to_string(index=False))

mapped_icds = [r['ICD'] for r in rows if r['mapped']]
print(f"\nICD-매핑된 코드 ({len(mapped_icds)}): {mapped_icds}")

## 2) PTB-XL + ZZU UMAP — 매핑된 5개 ICD 코드

각 모델 row × (combined / adult ≥18 / pediatric <18) 3 cols.
회색 = 다른 진단/그룹/미매핑. `balance_strict_groups=True` 로 adult vs pediatric 균등.

In [ ]:
fig, metrics = quick_dx(
    datasets=('ptbxl', 'zzu'),
    include_codes=mapped_icds,
    age_split=18.0,
    balance_strict_groups=True,
)

In [ ]:
# 모델·진단별 adult vs pediatric BACC 표
rows = []
for model, by_grp in metrics.items():
    a, p = by_grp.get('adult ≥18', {}), by_grp.get('pediatric <18', {})
    for code in mapped_icds:
        ba = a.get(code, {}).get('bacc')
        bp = p.get(code, {}).get('bacc')
        if ba is None or bp is None: continue
        rows.append({
            'model': model,
            'dx': ICD_DISPLAY.get(code, code),
            'code': code,
            'adult': round(ba, 3) if np.isfinite(ba) else None,
            'pedi':  round(bp, 3) if np.isfinite(bp) else None,
            'gap':   round(abs(ba-bp), 3) if (np.isfinite(ba) and np.isfinite(bp)) else None,
            'worst': round(min(ba, bp), 3) if (np.isfinite(ba) and np.isfinite(bp)) else None,
        })
df = pd.DataFrame(rows)
print(df.to_string(index=False))

## 3) chapman 데이터셋의 소아 분포 확인

In [ ]:
ages = get_ages('chapman')
fin = ages[np.isfinite(ages)]
print(f"chapman: N={len(ages):,}, age range={fin.min():.0f}~{fin.max():.0f}, mean={fin.mean():.1f}")
print(f"  pediatric (<18): {(fin < 18).sum():,}")
print(f"  adult    (≥18): {(fin >= 18).sum():,}")

## 4) chapman UMAP — 7개 raw 약자

ICD 통합 없이 chapman_paper_labels 컬럼 그대로. 한 줄 가로 legend.
(소아 비율이 1.4%라 single-axes 표시; 모델별 figure 4개)

In [ ]:
raw_cols = ['SR', 'ALS', 'APB', 'AF', 'SVT', 'TWC', 'ST']
for m in list_models():
    quick_dx_raw(m, 'chapman', label_columns=raw_cols, max_per_label=300)

## 5) (옵션) chapman 소아만 vs 성인만 — sample 분리

chapman ICD 매핑이 없어 quick_dx 의 age_split 기능을 그대로 못 씀.
수동으로 인덱스 분리하여 비교 figure 생성.

In [ ]:
import matplotlib.pyplot as plt
from scripts.umap_view import get_coords

# chapman 좌표 + age
for m in list_models()[:2]:  # 가독성을 위해 2개 모델만
    coords, sizes = get_coords(m, ['chapman'])
    n = sizes['chapman']
    a = ages[:n]
    adult_mask = np.isfinite(a) & (a >= 18)
    pedi_mask  = np.isfinite(a) & (a <  18)
    fig, axes = plt.subplots(1, 2, figsize=(10, 5), dpi=110)
    for ax, mask, title in [(axes[0], adult_mask, f'{m} · chapman adult ≥18 (n={int(adult_mask.sum()):,})'),
                            (axes[1], pedi_mask,  f'{m} · chapman pedi  <18 (n={int(pedi_mask.sum()):,})')]:
        rest = ~mask
        ax.scatter(coords[rest, 0], coords[rest, 1], c='#DDDDDD', s=3, alpha=0.2, linewidths=0)
        ax.scatter(coords[mask, 0], coords[mask, 1], c='#4477AA', s=4, alpha=0.6, linewidths=0)
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values(): sp.set_visible(False)
    plt.tight_layout(); plt.show()